# 03 - RFM Customer Segmentation

**E-Commerce Revenue & Customer Analytics**

RFM (Recency, Frequency, Monetary) segmentation groups customers by:
- **Recency**: days since their last order (lower = better)
- **Frequency**: number of orders placed
- **Monetary**: total revenue generated

Each dimension is scored 1-5 using quintiles, then combined into 8
business-friendly segments (Champions, Loyal Customers, Potential
Loyalists, New Customers, At Risk, Can't Lose Them, Hibernating, Lost).

The full scoring logic lives in `scripts/rfm_segmentation.py` -- this
notebook loads the pre-computed output and explores it visually.

Run `python scripts/run_pipeline.py` from the project root before executing this notebook.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROCESSED_DIR = os.path.join("..", "data", "processed")

In [ ]:
rfm = pd.read_csv(os.path.join(PROCESSED_DIR, "rfm_customer_segments.csv"))
segment_summary = pd.read_csv(os.path.join(PROCESSED_DIR, "rfm_segment_summary.csv"))

print(f"rfm: {rfm.shape}")
rfm.head()

## 1. Segment summary

In [ ]:
segment_summary

In [ ]:
SEGMENT_ORDER = [
    "Champions", "Loyal Customers", "Potential Loyalists", "New Customers",
    "At Risk", "Can't Lose Them", "Hibernating", "Lost",
]
segment_summary["segment"] = pd.Categorical(segment_summary["segment"], categories=SEGMENT_ORDER, ordered=True)
segment_summary = segment_summary.sort_values("segment")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

colors = sns.color_palette("Spectral", len(segment_summary))

axes[0].bar(segment_summary["segment"], segment_summary["customer_count"], color=colors)
axes[0].set_title("Customer Count by Segment", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Number of Customers")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(segment_summary["segment"], segment_summary["total_revenue"], color=colors)
axes[1].set_title("Revenue by Segment", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Total Revenue (Rs)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

**What does this tell the business?** This is the single most actionable chart in the project. Compare the two bars side by side: **Champions** are a modest share of the customer base but drive the majority of revenue -- exactly the customers retention budget should protect. **Hibernating** customers are the largest group by count but contribute comparatively little revenue -- a large, low-cost-per-customer re-engagement campaign (rather than expensive 1:1 outreach) is the right tool here.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(
    segment_summary["pct_of_revenue"],
    labels=segment_summary["segment"],
    autopct="%1.1f%%",
    startangle=90,
    colors=colors,
)
ax.set_title("Share of Total Revenue by RFM Segment", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Recency, Frequency, Monetary by segment

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].bar(segment_summary["segment"], segment_summary["avg_recency"], color="#C44E52")
axes[0].set_title("Avg. Recency (days) by Segment", fontsize=12, fontweight="bold")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(segment_summary["segment"], segment_summary["avg_frequency"], color="#55A868")
axes[1].set_title("Avg. Frequency (orders) by Segment", fontsize=12, fontweight="bold")
axes[1].tick_params(axis="x", rotation=45)

axes[2].bar(segment_summary["segment"], segment_summary["avg_revenue"], color="#4C72B0")
axes[2].set_title("Avg. Revenue (Rs) by Segment", fontsize=12, fontweight="bold")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

**What does this tell the business?** Champions and Loyal Customers cluster at low recency (bought recently) and high frequency/monetary. At Risk and Can't Lose Them show the inverse recency pattern (haven't bought in a while) while still holding relatively high frequency/monetary -- confirming they're valuable customers who are drifting away, not naturally low-value ones.

## 3. RFM heatmap: Frequency vs. Recency score, colored by average monetary value

In [ ]:
pivot = rfm.pivot_table(index="F_score", columns="R_score", values="monetary", aggfunc="mean")
pivot = pivot.sort_index(ascending=False)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(pivot, annot=True, fmt=",.0f", cmap="YlGnBu", ax=ax, cbar_kws={"label": "Avg. Monetary Value (Rs)"})
ax.set_title("Average Customer Value by Recency & Frequency Score", fontsize=14, fontweight="bold")
ax.set_xlabel("Recency Score (5 = most recent)")
ax.set_ylabel("Frequency Score (5 = most frequent)")
plt.tight_layout()
plt.show()

**What does this tell the business?** The top-right corner (high recency score, high frequency score) is the Champions zone and should show the highest average monetary value -- confirming the segmentation logic is internally consistent. This heatmap is also a quick way to spot-check the RFM scoring before trusting it for a marketing campaign.

## 4. Geographic concentration of high-value segments

In [ ]:
high_value = rfm[rfm["segment"].isin(["Champions", "Loyal Customers"])]
state_dist = high_value.groupby("state").size().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(state_dist.index, state_dist.values, color="#8172B2")
ax.set_title("Top 10 States: Champions + Loyal Customers", fontsize=14, fontweight="bold")
ax.set_xlabel("Number of Customers")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Priority action list: At Risk & Can't Lose Them (highest-value win-back targets)

In [ ]:
priority = rfm[rfm["segment"].isin(["At Risk", "Can't Lose Them"])].sort_values("monetary", ascending=False)
print(f"Customers in win-back priority segments: {len(priority):,}")
print(f"Combined revenue at stake: Rs {priority['monetary'].sum():,.2f}")
priority.head(15)[["customer_id", "customer_name", "city", "state", "recency", "frequency", "monetary", "segment"]]

**What does this tell the business?** These are customers who have historically spent well and ordered often but have gone quiet. Because they already have a proven purchase history (unlike a cold acquisition target), win-back campaigns targeted at this list typically have a much higher ROI than generic re-marketing.

## 6. Cross-check: unsupervised clustering (KMeans) vs. rule-based RFM segments

The segments above are rule-based (business-readable thresholds). As a sanity check, this section runs K-Means clustering directly on the scaled R/F/M values and compares cluster membership against the rule-based segments -- if the two broadly agree, that's good evidence the rule-based segments are capturing real structure in the data rather than being an arbitrary business narrative.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

features = rfm[["recency", "frequency", "monetary"]].copy()
scaled = StandardScaler().fit_transform(features)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm["kmeans_cluster"] = kmeans.fit_predict(scaled)

cluster_profile = rfm.groupby("kmeans_cluster").agg(
    customers=("customer_id", "count"),
    avg_recency=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean"),
).round(2)
cluster_profile

In [ ]:
cross_tab = pd.crosstab(rfm["kmeans_cluster"], rfm["segment"])
cross_tab

**What does this tell the business?** Each KMeans cluster should map fairly cleanly onto one or two of the rule-based RFM segments (e.g. the cluster with the lowest recency/highest frequency/monetary should overlap heavily with 'Champions'). Strong overlap validates that the rule-based segments -- which are much easier to explain to a marketing stakeholder than raw cluster IDs -- are a reasonable simplification of the underlying customer structure, not an arbitrary business narrative.

## Summary

RFM segmentation confirms a classic Pareto pattern: Champions and Loyal Customers together are a minority of the customer base but the majority of revenue. The At Risk / Can't Lose Them segments represent the highest-priority, highest-ROI win-back opportunity. The KMeans cross-check supports that the rule-based segments reflect genuine structure in customer behavior. See `reports/business_recommendations.md` for the full set of recommendations derived from this analysis.